In [1]:
file_path = 'myd.md'
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()
paragraphs = content.split('##')

for i, paragraph in enumerate(paragraphs):
    print(f'段落 {i + 1}:\n{paragraph}\n')
    print('-' * 20)

段落 1:
---
title: 'FAQ'
category: 'faq'
---

# 常见问题



--------------------
段落 2:
 1. openGauss 数据库开源许可协议是什么？

木兰宽松许可证 MulanPSL2 V2，无传染



--------------------
段落 3:
 2. openGauss 支持部署的硬件架构及操作系统有哪些？

ARM：openEuler 20.03LTS（推荐采用此操作系统）｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ FusionOS 22 ｜统信

X86：openEuler 20.03LTS ｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ CentOS 7.6 ｜ Asianux 7.6 ｜ FusionOS 22 ｜统信

<br/>

ubuntu / centos8 /centos10 / 红旗 需要适配编译数据库；在飞腾/海光 等服务器上安装需要重新适配编译。



--------------------
段落 4:
 3. openGauss 有哪些版本？

openGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：

1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；
2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；
3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；
4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。

详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。



--------------------
段落 5:
 4. openGauss 分布式部署方案是什么？

1. 基于 openLookeng 实现分布式分析能力，与 shardingsphere 配合 openGauss 组成 HTAP 数据库；
2. 基于分布式中间件 shardin

In [2]:
import ollama

def embedding(text):
    vector = ollama.embeddings(model="nomic-embed-text", prompt=text)
    return vector["embedding"]

text = "openGauss 是一款开源数据库"
emb = embedding(text)
dimensions = len(emb)
print("text :{}, embedding dim :{}, enbedding :{}...".format(text, dimensions, emb[:10]))

text :openGauss 是一款开源数据库, embedding dim :768, enbedding :[-0.5366359353065491, 1.3430017232894897, -3.522930383682251, -1.0018396377563477, -0.19522923231124878, 0.2820422053337097, -0.47223713994026184, 0.0804363563656807, -0.22050540149211884, -0.9979195594787598]...


In [28]:
import psycopg2

table_name = "opengauss_data"
schema_name = "user1_schema"
dimensions = 768

try:
    conn = psycopg2.connect(
        database="db1",
        user="user1",
        password="Password@2025!",
        host="127.0.0.1",
        port="5432"
    )
    
    with conn.cursor() as cur:
        cur.execute("SET search_path TO user1_schema, public;")
        cur.execute("SHOW search_path;")
        print(f"当前搜索路径: {cur.fetchone()[0]}")
        cur.execute("SELECT EXISTS(SELECT 1 FROM pg_namespace WHERE nspname = %s);", (schema_name,))
        if not cur.fetchone()[0]:
            cur.execute(f"CREATE SCHEMA {schema_name} AUTHORIZATION user1;")
            conn.commit()
            print(f"模式 {schema_name} 创建成功！")
        
        # 创建表
        cur.execute(
            f"""
            CREATE TABLE {table_name} (  -- 无需指定 schema，已在搜索路径中
                id INT PRIMARY KEY,
                content TEXT,
                emb VECTOR({dimensions})
            );
            """
        )
        conn.commit()
        print(f"表 {schema_name}.{table_name} 创建成功！")

except psycopg2.Error as e:
    print(f"数据库错误: {e}")
    conn.rollback()
finally:
    if conn:
        conn.close()

当前搜索路径: user1_schema, public
模式 user1_schema 创建成功！
表 user1_schema.opengauss_data 创建成功！


In [31]:
import psycopg2

table_name = "opengauss_data"
conn = psycopg2.connect(
    database="db1",
    user="user1",
    password="Password@2025!",
    host="127.0.0.1",
    port="5432"
)

try:
    with conn.cursor() as cur:
        for i, paragraph in enumerate(paragraphs):
            emb = embedding(paragraph)
            insert_data_sql = f'''INSERT INTO {table_name} (id, content, emb) VALUES (%s, %s, %s);'''
            cur.execute(insert_data_sql, (i, paragraph, emb))
        conn.commit()
        cur.execute("CREATE INDEX ON {} USING hnsw (emb vector_l2_ops);".format(table_name))
        conn.commit()
        
except Exception as e:
    print(f"操作数据库时出错: {e}")
    conn.rollback()
finally:
    if conn:
        conn.close()

In [47]:
question = "openGauss 发布了哪些版本？"

In [48]:
import psycopg2

table_name = "opengauss_data"
conn = psycopg2.connect(
    database="db1",
    user="user1",
    password="Password@2025!",
    host="127.0.0.1",
    port="5432"
)

emb_data = embedding(question)  # 问题数据向量化
dimensions = len(emb_data)

cur = conn.cursor()
cur.execute("select content from {} order by emb <-> '{}' limit 1;".format(table_name, emb_data))
conn.commit()

rows = cur.fetchall()
print(rows)

cur.close()
conn.close()

[(' 3. openGauss 有哪些版本？\n\nopenGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：\n\n1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；\n2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；\n3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；\n4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。\n\n详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。\n\n',)]


In [49]:
import psycopg2

table_name = "opengauss_data"
conn = psycopg2.connect(
    database="db1",
    user="user1",
    password="Password@2025!",
    host="127.0.0.1",
    port="5432"
)

emb_data = embedding(question)
dimensions = len(emb_data)

cur = conn.cursor()
cur.execute("select content from {} order by emb <-> '{}' limit 1;".format(table_name, emb_data))
conn.commit()

rows = cur.fetchall()
print(rows)

cur.close()
conn.close()

[(' 3. openGauss 有哪些版本？\n\nopenGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：\n\n1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；\n2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；\n3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；\n4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。\n\n详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。\n\n',)]


In [50]:
SYSTEM_PROMPT = "你作为一个对话AI助手，结合上下文信息简练高效的回答用户提出的问题"
USER_PROMPT = f"请回答{question}的问题，不需要额外的无用回答"

In [51]:
#直接使用大模型
response = ollama.chat(
    model="deepseek-r1",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ]
)
print(response["message"]["content"])

<think>
好，用户问的是openGauss有哪些版本。我需要结合提供的上下文信息来回答。

根据内容，openGauss分为四个版本：企业版、极简版、轻量版和分布式镜像版。每个版本有不同的特点和适用场景，比如企业版适合企业用户，而极简版适合个人开发者。

所以，答案应该列出这四个版本，并简洁说明每种版本的用途。
</think>

openGauss 发布了以下几个版本：

1. 企业版：具备更齐全的集群管理功能，适合企业用户；
2. 极简版：安装配置简单，解压即可使用，适合个人开发者；
3. 轻量版：精简功能，缩减安装包大小，内存占用更少；
4. 分布式镜像版：基于 ShardingSphere 和 k8s 的分布式容器化镜像。


In [56]:
context = "\n".join(row[0] for row in rows)
SYSTEM_PROMPT = "你作为一个对话AI助手，结合上下文信息简练高效的回答用户提出的问题"
USER_PROMPT = f"请结合{context}信息来回答{question}的问题，不需要额外的无用回答"

In [57]:
#RAG
response = ollama.chat(
    model="deepseek-r1",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ]
)
print(response["message"]["content"])

<think>
嗯，用户问的是“openGauss 发布了哪些版本？”，我需要根据提供的上下文信息来回答。首先，文本里提到openGauss每两年发布一个LTS版本，并且半年发布创新版本。然后有四个特定的企业版、极简版、轻量版和分布式镜像版。

我应该把所有这些版本列出来，用简洁的方式呈现，可能分点说明每个版本的特点会更清楚。比如企业版适合企业用户，极简版适合个人开发者，轻量版精简功能，分布式镜像基于特定框架。

同时，要确保回答直接，没有额外信息，符合用户的简练需求。所以，整合这些信息，用明确的标题和分点描述应该就能满足要求了。
</think>

开放高斯（openGauss）发布了多个版本，具体包括以下版本：

1. **LTS 版本**：每两年发布一次，作为长期支持版本，适合大规模部署。

2. **创新版本**：半年发布，旨在供用户进行联合创新测试使用。

3. **企业版（openGauss 企业版）**：提供全面的集群管理功能，适用于企业用户。

4. **极简版（openGauss 极简版）**：安装配置简单，解压即可使用，适合个人开发者。

5. **轻量版（openGauss 轻量版）**：精简功能模块，减少安装包大小和内存占用，适用于对资源要求较低的场景。

6. **分布式镜像版（openGauss 分布式镜像）**：基于 ShardingSphere 和 Kubernetes 的分布式容器化镜像，适合容器化部署。
